# TopK SAE on a language model

This notebook trains a sparse autoencoder on the residual stream of
**Pythia-70m-deduped** (EleutherAI, d = 512) using the first 10k documents
of the Pile (`NeelNanda/pile-10k`), then analyses what it learned.

The workflow mirrors the vision notebook:

1. **Load a pretrained model and text**; tokenize into fixed-length chunks.
2. **Harvest activations** at one layer, drop position 0 of every chunk
   (its norm is far larger than other tokens'), shuffle, **cache to disk**.
3. **Train the SAE**, 4. **evaluate** (FVE, KL to the clean next-token
   distribution), 5. optionally **KL fine-tune**, 6. **save / load**.
7. **Analyse the dictionary**: density, redundancy, the tokens each latent
   promotes (logit lens on decoder atoms).
8. **Inspect latents**: top-activating contexts, highlighted per token.
9. **Attribute** a next-token prediction to latents and tokens.

Install: `uv sync --extra notebooks`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import (
    DataLoader,
    TensorDataset
)

from innards import RowLoader, RowStream, write_rows
from innards.sae import (
    KLFinetuneConfig,
    SAETrainer,
    SAE,
    TrainerConfig,
)
import torch.nn.functional as F

from innards.analysis import geometry, interp, plot
from innards.sae import analysis as A
from innards.sae import viz

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTOCAST = torch.bfloat16 if DEVICE == "cuda" else None

MODEL_NAME = "EleutherAI/pythia-70m-deduped"
LAYER = 3 # residual stream after layer 3 of 6
CTX = 128 # tokens per chunk
MAX_TOKENS = 1_000_000
CACHE_DIR = Path("activations") / f"pythia-70m-deduped_layer{LAYER}"
OUT_DIR = Path("checkpoints") / f"pythia-70m-deduped_layer{LAYER}"
BATCH_SEQS = 32
EXCLUDE = (0,)
print(f"device={DEVICE}")

device=cpu, smoke_test=False


## 1. Model and data

Documents are tokenized, joined with an end-of-text token and cut into
`CTX`-token chunks, so there is no padding to mask. (With padded batches,
pass `mask_fn=lambda batch: batch["attention_mask"]` to the `RowStream` and
the analysis functions instead.)

In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
texts = load_dataset("NeelNanda/pile-10k", split="train")["text"]

model = model.eval().to(DEVICE)
layer = model.gpt_neox.layers[LAYER]


def chunk_texts(texts, ctx: int, max_tokens: int) -> torch.Tensor:
    ids: list[int] = []
    for doc in tokenizer(list(texts))["input_ids"]:
        ids.extend(doc + [tokenizer.eos_token_id])
        if len(ids) >= max_tokens:
            break
    n = min(len(ids), max_tokens) // ctx
    return torch.tensor(ids[: n * ctx]).view(n, ctx)


chunks = chunk_texts(texts, CTX, MAX_TOKENS)
n_val = max(len(chunks) // 20, 4)
train_ids, val_ids = chunks[n_val:], chunks[:n_val]
train_loader = DataLoader(TensorDataset(train_ids), batch_size=BATCH_SEQS, shuffle=True)
val_loader = DataLoader(TensorDataset(val_ids), batch_size=BATCH_SEQS, shuffle=False)
print(f"train chunks: {len(train_ids):,} ({train_ids.numel():,} tokens), val chunks: {len(val_ids):,}")

/Users/erik/PycharmProjects/sae/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 12903.46it/s]


train chunks: 7,422 (950,016 tokens), val chunks: 390


## 2. Harvest and cache activations

~9.5M tokens at d = 512 in bf16 is ~10 GB on disk. Lower `MAX_TOKENS` for a
quicker run.

In [3]:
stream = RowStream(model, train_loader, layer, exclude=EXCLUDE, device=DEVICE, dtype=torch.bfloat16,
                   autocast_dtype=AUTOCAST)
if not (CACHE_DIR / "meta.json").exists():
    write_rows(stream, CACHE_DIR, dtype=torch.bfloat16,
               metadata={"model": MODEL_NAME, "layer": f"gpt_neox.layers.{LAYER}", "ctx": CTX,
                         "exclude": list(EXCLUDE)})
train_acts = RowLoader(CACHE_DIR, device=DEVICE)
print(f"cached tokens: {train_acts.n_rows:,}")

cached tokens: 942,080


## 3. Train the SAE

8x expansion (4096 latents), k = 32, BatchTopK, one epoch over the cache.

In [4]:
D_MODEL = model.config.hidden_size
sae = SAE(d_in=D_MODEL, d_sae=8 * D_MODEL, k=32, activation="batchtopk",
              dead_tokens_threshold=1_000_000)
trainer = SAETrainer(sae, TrainerConfig(
    num_epochs=1,
    loggers=("tensorboard",),
    run_name=f"pythia-70m-layer{LAYER}",
    output_dir=str(OUT_DIR),
    log_every=50,
))
trainer.train(train_acts)

step     200 | l0 32 | aux_loss 0 | loss 0.3783 | fve 0.6217 | dense_frac 0.002197 | dead_frac 0 | lr 0.0002609 | k 32
step     230 | l0 32 | aux_loss 0 | loss 0.2445 | fve 0.7555 | dense_frac 0.005127 | dead_frac 0 | lr 0 | k 32


TopKSAE()

## 4. Evaluate

The downstream KL compares next-token distributions at every position with
and without the SAE spliced in.

In [5]:
val_stream = RowStream(model, val_loader, layer, exclude=EXCLUDE, device=DEVICE, autocast_dtype=AUTOCAST)
kl_cfg = KLFinetuneConfig(
    steps=100,
    exclude=EXCLUDE,
    output_fn=lambda out: out.logits,
    log_every=10,
)
print(trainer.evaluate(val_stream))
print(trainer.evaluate_downstream(model, layer, val_loader, kl_cfg))

{'eval/nmse': 0.3764991970272529, 'eval/fve': 0.6235008029727471, 'eval/l0': 40.270936667349865}
{'downstream/kl': 1.5486696224946241, 'downstream/top1_agreement': 0.41440304487179486}


## 5. KL fine-tuning (optional)

In [6]:
RUN_KL_FINETUNE = True
if RUN_KL_FINETUNE:
    trainer.finetune_kl(model, layer, train_loader, kl_cfg)
    print(trainer.evaluate(val_stream))
    print(trainer.evaluate_downstream(model, layer, val_loader, kl_cfg))

## 6. Save and load

In [7]:
path = trainer.save(OUT_DIR / "sae_final.safetensors", model=MODEL_NAME, hook=f"gpt_neox.layers.{LAYER}")
sae = SAE.load(path, device=DEVICE)
print(SAE.read_metadata(path)["hook"], sae.cfg)

gpt_neox.layers.3 SAEConfig(d_in=512, d_sae=4096, k=32, as_batchtopk=True, abs_topk=False, k_aux=256, aux_coeff=0.03125, dead_tokens_threshold=1000000, threshold_ema_decay=0.99, multi_topk=False, multi_topk_factor=4, multi_topk_coeff=0.125, matryoshka=False, matryoshka_prefixes=None, archetypal=False, archetypal_n_candidates=None, archetypal_delta=1.0, archetypal_learn_scale=True, loss_normalization='batch')


## 7. The dictionary as a whole

In [ ]:
res = A.analyze_dataset(sae, model, layer, val_loader, n_top=20, exclude=EXCLUDE)
atoms = F.normalize(sae.decoder_weight().detach(), dim=-1)  # the dictionary as unit directions
max_cos, _ = geometry.nearest(atoms, k=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
plot.histogram(res.stats.frequency, log10=True, xlabel="log10 firing frequency", ylabel="latents",
               title="latent density", ax=axes[0])
plot.histogram(max_cos, xlabel="max cosine similarity", ylabel="latents", title="decoder redundancy", ax=axes[1])
plot.embedding(atoms, method="pca", values=torch.log10(res.stats.frequency + 1e-6),  # "umap" with the umap extra
               values_label="log10 frequency", title="dictionary map", ax=axes[2])
plt.show()
print(f"dead: {len(res.stats.dead)}, dense (>10%): {len(res.stats.dense())}")

**Logit lens on decoder atoms**: projecting an atom through the unembedding
shows which next tokens the latent directly promotes and suppresses. It
skips the remaining layers and the final LayerNorm, so it is most reliable
for later layers.

In [ ]:
tok = lambda i: repr(tokenizer.decode([int(i)]))
unembed = model.get_output_embeddings().weight

alive = (res.stats.frequency > 1e-4).nonzero().squeeze(-1)
latents = alive[torch.randperm(len(alive), generator=torch.Generator().manual_seed(0))[:4]].tolist()
top_v, top_i, bot_v, bot_i = interp.top_outputs(interp.readout_effect(atoms[latents], unembed), n=6)
fig, axes = plt.subplots(1, len(latents), figsize=(5 * len(latents), 4), layout="constrained")
for j, lat in enumerate(latents):
    ids = torch.cat([top_i[j], bot_i[j].flip(0)])
    plot.bars(torch.cat([top_v[j], bot_v[j].flip(0)]), [tok(i) for i in ids], xlabel="direct effect",
              title=f"latent {lat}", ax=axes[j])
plt.show()

## 8. Latent dashboards: top-activating contexts

Each row is a validation chunk around the token where the latent fires
most (underlined); colour is the latent's activation on every token.

In [ ]:
def show_latent(latent: int, n: int = 10, context: int = 15):
    examples = res.top.unique_examples(latent)[:n]
    if not examples:
        print(f"latent {latent} never fired")
        return
    rows = []
    for _, e, p in examples:
        ids = val_ids[e]
        acts = A.encode_inputs(sae, model, layer, ids[None].to(DEVICE))[0, :, latent]
        acts[list(EXCLUDE)] = 0
        rows.append(([tokenizer.decode([int(t)]) for t in ids], acts.cpu(), p))
    title = f"latent {latent}: frequency {res.stats.frequency[latent]:.3%}, " \
            f"promotes {', '.join(tok(i) for i in interp.readout_effect(atoms[latent], unembed)[0].topk(5).indices)}"
    viz.show_html(viz.text_examples_html(rows, title=title, context=context))


for lat in latents:
    show_latent(lat)

## 9. Attribution: which latents drive a prediction?

Attribute the logit of a chosen next token at the last position to every
latent at every earlier token (attribution patching with the SAE error
term). The highlighted text shows, per token, the total attribution of the
top latent.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
target = " Paris"
ids = torch.tensor([tokenizer.encode(prompt)], device=DEVICE)
target_id = tokenizer.encode(target)[0]

attr = A.latent_attribution(sae, model, layer, ids, lambda out, _: out.logits[:, -1, target_id],
                            exclude=EXCLUDE)
values, top = attr.top_latents(0, n=8)
plot.bars(torch.cat([values, attr.error_attribution[0].sum()[None]]),
          [f"#{int(l)}" for l in top] + ["SAE error"], xlabel="attribution",
          title=f"logit of {target!r} after the prompt")
plt.show()

tokens = [tokenizer.decode([int(t)]) for t in ids[0]]
full = attr.attribution[0, :, top[0]].cpu()  # zero at unpatched positions
viz.show_html(f"<b>latent {int(top[0])}</b>: attribution per token" + viz.token_activation_html(tokens, full))